# 🎯 Fine-Tuning the Bug Classifier (Glitch Investigator AI)

Fine-tunes a small **DistilBERT** model to classify a Python snippet into one of six bug categories:
`state_bug, logic_inverted, missing_validation, off_by_one, dead_control, clean`.

**Run this on Google Colab with a GPU** (Runtime → Change runtime type → GPU).

Steps: install deps → upload `data/bugs.csv` → tokenize → train → evaluate (loss curve + confusion matrix) → save & download the model into `model/`.

In [ ]:
# 1. Install dependencies
!pip -q install "transformers>=4.40.0" "datasets>=2.19.0" "scikit-learn>=1.3.0" "torch>=2.0.0" pandas matplotlib

## 2. Load the dataset

Upload the `bugs.csv` produced locally by `python3 data/make_dataset.py`.

In [ ]:
import pandas as pd

try:
    from google.colab import files  # Colab: prompt an upload
    uploaded = files.upload()       # choose bugs.csv
    csv_path = next(iter(uploaded))
except Exception:
    csv_path = '../data/bugs.csv'    # running locally

df = pd.read_csv(csv_path)
print(df.shape)
df['label'].value_counts()

In [ ]:
# 3. Encode labels and make a stratified train/test split
from sklearn.model_selection import train_test_split

LABELS = ['state_bug', 'logic_inverted', 'missing_validation',
          'off_by_one', 'dead_control', 'clean']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

df['y'] = df['label'].map(label2id)
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['y'], random_state=42)
print('train:', len(train_df), 'test:', len(test_df))

In [ ]:
# 4. Tokenize
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    return tokenizer(batch['snippet'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['snippet', 'y']]).map(tok, batched=True)
test_ds = Dataset.from_pandas(test_df[['snippet', 'y']]).map(tok, batched=True)
train_ds = train_ds.rename_column('y', 'labels')
test_ds = test_ds.rename_column('y', 'labels')
cols = ['input_ids', 'attention_mask', 'labels']
train_ds.set_format('torch', columns=cols)
test_ds.set_format('torch', columns=cols)

In [ ]:
# 5. Fine-tune
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_macro': f1_score(labels, preds, average='macro')}

args = TrainingArguments(
    output_dir='out',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='no',
    report_to='none')

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=test_ds, compute_metrics=metrics)
trainer.train()

In [ ]:
# 6. Loss curve (save the figure for your model card)
import matplotlib.pyplot as plt

hist = trainer.state.log_history
tr = [(h['epoch'], h['loss']) for h in hist if 'loss' in h]
ev = [(h['epoch'], h['eval_loss']) for h in hist if 'eval_loss' in h]
plt.figure(figsize=(6, 4))
if tr:
    plt.plot(*zip(*tr), marker='o', label='train loss')
if ev:
    plt.plot(*zip(*ev), marker='o', label='eval loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Training curve')
plt.tight_layout(); plt.savefig('loss_curve.png', dpi=120); plt.show()

In [ ]:
# 7. Confusion matrix + per-class report (both go in the model card)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print(classification_report(y_true, y_pred, target_names=LABELS))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=120); plt.show()

In [ ]:
# 8. Save the model + label map, then download it into your local model/ folder
import json, shutil

SAVE_DIR = 'model'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f'{SAVE_DIR}/labels.json', 'w') as f:
    json.dump(LABELS, f)

shutil.make_archive('model', 'zip', SAVE_DIR)
print('Saved. Download model.zip and unzip its contents into your repo model/ folder.')
try:
    from google.colab import files
    files.download('model.zip')
except Exception:
    pass